In [2]:
import pandas as pd

Vietnamese_data = pd.read_excel("Vietnamese_reviews.xlsx")
print(Vietnamese_data.shape)
Vietnamese_data.head()


(19938, 12)


,Company Name,Cmt_day,Title,What I liked,Suggestions for improvement,Rating,Salary & benefits,Training & learning,Management cares about me,Culture & fun,Office & workspace,Recommend?
0,Success Software Services,2024-06-01,Phúc lợi khá OK còn công việc thì tùy dự án và...,"Văn phòng khá đẹp, nhiều event trong nội bộ cô...",Nên cải thiện quy trình và phúc lợi dành cho o...,3,3,3,3,3,3,Yes
1,Success Software Services,2023-09-01,"Môi trường năng động, trẻ, chính sách bảo mật ...","Môi trường năng động, trẻ trung.\nCó cơ hội là...",Tăng số ngày nghỉ phép như 12 phép năm + 3 phé...,4,4,4,3,3,4,Yes
2,Success Software Services,2021-07-01,Một nơi tốt để học và phát triểu bản thân,"Định hướng công ty tốt, khách hàng châu mỹ nên...","Cần đầu tư hơn về cơ sở vật chất, văn phòng , ...",5,4,5,5,4,3,Yes
3,Success Software Services,2024-03-01,"Review mang tính góp ý, không có ý định chê cô...","Đồng nghiệp thân thiện, hoà đồng, hỗ trợ hết m...","- Ràng buộc thực tập sinh 2 năm, nhưng lúc phỏ...",4,2,2,4,4,3,No
4,Success Software Services,2024-02-01,"Cty trẻ, năng động, lead có tâm, benefit ít.","Cty trẻ, năng động, lead có tâm. Process khá c...",Nên thêm các benefit tương ứng khi member làm ...,3,2,3,3,2,4,No


In [3]:
import re

def clean_review(text):
    if not isinstance(text, str) or not text:
        return ""

    # BƯỚC 1: Xử lý các biến thể của xuống dòng để split chính xác
    text = text.replace('\\n', '\n')
    
    # BƯỚC 2: Xử lý Emoji/Emoticons trước khi chúng bị băm nhỏ bởi regex ký tự đặc biệt
    text = re.sub(r':[Dd]+\b', ' ', text)
    text = re.sub(r'(😕)+', ' ', text)
    text = re.sub(r':\(+', ' ', text)
    text = re.sub(r'(😊)+', ' ', text)

    # BƯỚC 3: Tách dữ liệu thành từng dòng (List of lines)
    lines = text.split('\n')
    cleaned_lines = []

    # BƯỚC 4: Duyệt qua từng dòng để xóa sạch dấu gạch, số, sao...
    for line in lines:
        line = line.strip()
        if not line:
            continue

        line = re.sub(r'^[ \t\-\*\+\•\d\.\)\(\[\]]+', '', line)
        line = re.sub(r'\.{2,}', '', line)
        line = re.sub(r',\s*$', '', line)
        line = re.sub(r'\s+', ' ', line).strip()

        if line:
            if not re.search(r'[.!?]$', line):
                line += '.'
            cleaned_lines.append(line)

    # BƯỚC 5: Loại bỏ duplicate sentences (giữ thứ tự, không phân biệt hoa thường)
    seen = set()
    unique_lines = []
    for line in cleaned_lines:
        key = line.lower().strip()
        if key not in seen:
            seen.add(key)
            unique_lines.append(line)
    
    result = ' '.join(unique_lines)
    result = re.sub(r'\.+', '.', result)
    result = re.sub(r'\s\.', '.', result)
    result = re.sub(r'\s+', ' ', result)
    
    return result.strip()


In [4]:
Vietnamese_data["Title"] = Vietnamese_data["Title"].astype("str").apply(clean_review)
Vietnamese_data["What I liked"] = Vietnamese_data["What I liked"].astype("str").apply(clean_review)
Vietnamese_data["Suggestions for improvement"] = Vietnamese_data["Suggestions for improvement"].astype("str").apply(clean_review)

Vietnamese_data.head()


,Company Name,Cmt_day,Title,What I liked,Suggestions for improvement,Rating,Salary & benefits,Training & learning,Management cares about me,Culture & fun,Office & workspace,Recommend?
0,Success Software Services,2024-06-01,Phúc lợi khá OK còn công việc thì tùy dự án và...,"Văn phòng khá đẹp, nhiều event trong nội bộ cô...",Nên cải thiện quy trình và phúc lợi dành cho o...,3,3,3,3,3,3,Yes
1,Success Software Services,2023-09-01,"Môi trường năng động, trẻ, chính sách bảo mật ...","Môi trường năng động, trẻ trung. Có cơ hội làm...",Tăng số ngày nghỉ phép như 12 phép năm + 3 phé...,4,4,4,3,3,4,Yes
2,Success Software Services,2021-07-01,Một nơi tốt để học và phát triểu bản thân.,"Định hướng công ty tốt, khách hàng châu mỹ nên...","Cần đầu tư hơn về cơ sở vật chất, văn phòng , ...",5,4,5,5,4,3,Yes
3,Success Software Services,2024-03-01,"Review mang tính góp ý, không có ý định chê cô...","Đồng nghiệp thân thiện, hoà đồng, hỗ trợ hết m...","Ràng buộc thực tập sinh 2 năm, nhưng lúc phỏng...",4,2,2,4,4,3,No
4,Success Software Services,2024-02-01,"Cty trẻ, năng động, lead có tâm, benefit ít.","Cty trẻ, năng động, lead có tâm. Process khá c...",Nên thêm các benefit tương ứng khi member làm ...,3,2,3,3,2,4,No


In [5]:
# BƯỚC 7: Xóa duplicate cho từng cột (giữ lần đầu, các lần sau để trống)
print("\nĐang xóa duplicate...")
cols = ["Title", "What I liked", "Suggestions for improvement"]
for col in cols:
    mask = Vietnamese_data[col].duplicated(keep="first") & (Vietnamese_data[col].str.strip() != "")
    Vietnamese_data.loc[mask, col] = ""

print("\nSố dòng có giá trị sau khi xử lý:")
for col in cols:
    print(f"  {col}: {(Vietnamese_data[col].str.strip() != '').sum()} dòng")

Vietnamese_data.head(10)


Đang xóa duplicate...

Số dòng có giá trị sau khi xử lý:
  Title: 15446 dòng
  What I liked: 19852 dòng
  Suggestions for improvement: 17977 dòng


,Company Name,Cmt_day,Title,What I liked,Suggestions for improvement,Rating,Salary & benefits,Training & learning,Management cares about me,Culture & fun,Office & workspace,Recommend?
0,Success Software Services,2024-06-01,Phúc lợi khá OK còn công việc thì tùy dự án và...,"Văn phòng khá đẹp, nhiều event trong nội bộ cô...",Nên cải thiện quy trình và phúc lợi dành cho o...,3,3,3,3,3,3,Yes
1,Success Software Services,2023-09-01,"Môi trường năng động, trẻ, chính sách bảo mật ...","Môi trường năng động, trẻ trung. Có cơ hội làm...",Tăng số ngày nghỉ phép như 12 phép năm + 3 phé...,4,4,4,3,3,4,Yes
2,Success Software Services,2021-07-01,Một nơi tốt để học và phát triểu bản thân.,"Định hướng công ty tốt, khách hàng châu mỹ nên...","Cần đầu tư hơn về cơ sở vật chất, văn phòng , ...",5,4,5,5,4,3,Yes
3,Success Software Services,2024-03-01,"Review mang tính góp ý, không có ý định chê cô...","Đồng nghiệp thân thiện, hoà đồng, hỗ trợ hết m...","Ràng buộc thực tập sinh 2 năm, nhưng lúc phỏng...",4,2,2,4,4,3,No
4,Success Software Services,2024-02-01,"Cty trẻ, năng động, lead có tâm, benefit ít.","Cty trẻ, năng động, lead có tâm. Process khá c...",Nên thêm các benefit tương ứng khi member làm ...,3,2,3,3,2,4,No
5,Success Software Services,2021-07-01,Khách hàng chủ yếu thị trường Mỹ nên môi trườn...,Có thưởng dự án mỗi quý. Sếp luôn quan tâm đến...,Công ty cần tăng tỉ lệ đóng BHXH cho nhân viên...,5,5,5,5,5,4,Yes
6,Success Software Services,2019-02-01,Hợp để fresher lấy kinh nghiệm.,"Môi trường cởi mở, đầu vào không quá khó, rất ...",,4,3,4,3,4,3,Yes
7,Success Software Services,2018-11-01,Môi trường phát triển tốt.,"Cơ hội giao tiếp bằng tiếng anh, làm việc với ...",Resource utilization phải làm nhiều project cù...,3,3,3,2,2,3,Yes
8,Success Software Services,2017-06-01,"Môi trường năng động, phù hợp cho sinh viên mớ...",Môi trường năng động. Sếp thân thiện. Khá open...,Dự án gấp OT hơi nhiều. Lương thưởng phúc lợi ...,4,3,4,3,4,3,Yes
9,Success Software Services,2016-11-01,Môi trường khá tốt.,Mô trường tương đối thân thiện. Quy trình ổn. ...,Văn phòng và cách sắp xếp bàn chưa thoải mái lắm.,4,3,4,4,4,3,Yes


In [25]:
Vietnamese_data.to_excel("Vietnamese_reviews_cleaned.xlsx", index=False)
print("Đã xuất file Vietnamese_reviews_cleaned.xlsx")

Đã xuất file Vietnamese_reviews_cleaned.xlsx


In [6]:
# Gộp 3 cột lại thành 1 cột cho 100 dòng đầu
# Vietnamese_data_100 = Vietnamese_data.head(100).copy()

# Tạo cột mới gộp 3 cột lại, ngăn cách bởi khoảng trắng
Vietnamese_data["Combined_Review"] = (
    Vietnamese_data["Title"].fillna("").astype(str).str.strip() + " " +
    Vietnamese_data["What I liked"].fillna("").astype(str).str.strip() + " " +
    Vietnamese_data["Suggestions for improvement"].fillna("").astype(str).str.strip()
).str.strip()

# Loại bỏ khoảng trắng thừa
Vietnamese_data["Combined_Review"] = Vietnamese_data["Combined_Review"].str.replace(r'\s+', ' ', regex=True)

print(f"Đã gộp 3 cột thành 1 cột 'Combined_Review' cho {len(Vietnamese_data)} dòng đầu")
print(f"\nĐộ dài trung bình: {Vietnamese_data['Combined_Review'].str.len().mean():.0f} ký tự")

Vietnamese_data[["Combined_Review"]].head(10)

Đã gộp 3 cột thành 1 cột 'Combined_Review' cho 19938 dòng đầu

Độ dài trung bình: 384 ký tự


,Combined_Review
0,Phúc lợi khá OK còn công việc thì tùy dự án và...
1,"Môi trường năng động, trẻ, chính sách bảo mật ..."
2,Một nơi tốt để học và phát triểu bản thân. Địn...
3,"Review mang tính góp ý, không có ý định chê cô..."
4,"Cty trẻ, năng động, lead có tâm, benefit ít. C..."
5,Khách hàng chủ yếu thị trường Mỹ nên môi trườn...
6,Hợp để fresher lấy kinh nghiệm. Môi trường cởi...
7,Môi trường phát triển tốt. Cơ hội giao tiếp bằ...
8,"Môi trường năng động, phù hợp cho sinh viên mớ..."
9,Môi trường khá tốt. Mô trường tương đối thân t...


In [ ]:
import re
from collections import Counter
import pandas as pd
import nltk
nltk.download('words', quiet=True)
from nltk.corpus import words

english_vocab = set(words.words())
# Blacklist: từ tiếng Việt không dấu trùng với tiếng Anh
VIETNAMESE_BLACKLIST = {
    'co', 'con', 'ban', 'chi', 'cho', 'chua', 'chu', 'cua', 'da', 'dan',
    'dao', 'den', 'di', 'do', 'doi', 'du', 'duoc', 'giao', 'gio', 'hai',
    'hay', 'het', 'ho', 'hoc', 'khi', 'khong', 'la', 'lam', 'lon', 'ma',
    'mai', 'mat', 'met', 'moi', 'nam', 'nao', 'nay', 'nha', 'nhi', 'nho',
    'nhu', 'nhung', 'noi', 'qua', 'quan', 'rat', 'roi', 'sau', 'se', 'tai',
    'tam', 'than', 'the', 'thi', 'thoi', 'thu', 'tot', 'trang', 'tre', 'tren',
    'trong', 'tu', 'va', 've', 'vi', 'voi', 'xa', 'xin', 'xu', 'ca', 'cac',
    'anh', 'chi', 'em', 'ong', 'ba', 'de', 'dong', 'long', 'tin', 'tri', 'sinh',
}
# def extract_english_words_strict(text):
#     words_found = re.findall(r'\b[a-zA-Z]{3,}\b', str(text).lower())
#     # Chỉ giữ từ có trong English dictionary
#     english_words = [
#         w for w in words_found 
#         if w in english_vocab and w not in VIETNAMESE_BLACKLIST
#     ]
#     return english_words
import enchant

d = enchant.Dict("en_US")
def extract_english_words_enchant(text):
    words_found = re.findall(r'\b[a-zA-Z]{4,}\b', str(text).lower())
    return [w for w in words_found if d.check(w) and w not in VIETNAMESE_BLACKLIST]

# Tổng hợp tất cả từ tiếng Anh vào một danh sách
all_words = []
for review in Vietnamese_data["Combined_Review"]: # full_content là cột văn bản đã gộp
    all_words.extend(extract_english_words_enchant(review))

# Thống kê tần suất
word_counts = Counter(all_words)
# Xem top 50 từ xuất hiện nhiều nhất
print(len(word_counts), "từ tiếng Anh được tìm thấy sau khi lọc:")
for word, count in word_counts.most_common(50):
    print(f"{word}: {count}")

False

In [34]:
Vietnamese_data_100.to_excel("Vietnamese_100_cleaned.xlsx", index=False)
print("Đã xuất file Vietnamese_100_cleaned.xlsx")

Đã xuất file Vietnamese_100_cleaned.xlsx
